In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print(OPENAI_API_KEY[:2])

UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
print(UPSTAGE_API_KEY[30:])

In [17]:
import warnings
warnings.filterwarnings("ignore")

import uuid
import re
import json
from typing import List, Literal

from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage
from langchain.agents import tool
from langchain_upstage import UpstageEmbeddings, ChatUpstage
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState

In [18]:
embeddings_model = UpstageEmbeddings(model="solar-embedding-1-large")

menu_db = FAISS.load_local(
    "../db/cafe_db",
    embeddings_model,
    allow_dangerous_deserialization=True
)

llm = ChatUpstage(
        model="solar-pro",
        base_url="https://api.upstage.ai/v1",
        temperature=0.5
)

In [19]:
def extract_menu_info(doc: Document) -> dict:
    """Vector DB 문서에서 구조화된 메뉴 정보 추출"""
    content = doc.page_content
    menu_name = doc.metadata.get('menu_name', 'Unknown')
    
    # 정규표현식으로 가격, 설명 등 추출
    price_match = re.search(r'₩([\d,]+)', content)
    description_match = re.search(r'설명:\s*(.+?)(?:\n|$)', content, re.DOTALL)
    
    return {
        "name": menu_name,
        "price": price_match.group(0) if price_match else "가격 정보 없음",
        "description": description_match.group(1).strip() if description_match else "설명 없음"
    }

In [20]:
@tool
def search_menu(user_message: str) -> List[Document]:
    """Uses the user's message directly to search for menu information."""
    docs = menu_db.similarity_search(user_message, k=4)
    # if len(docs) > 0:
    #     return docs
    
    # return [Document(page_content="관련 메뉴 정보를 찾을 수 없습니다.")]
    if len(docs) <= 0:
        return [Document(page_content="관련 메뉴 정보를 찾을 수 없습니다.")]

    structured_results = [extract_menu_info(doc) for doc in docs]
    
    return json.dumps(structured_results, ensure_ascii=False, indent=2)

@tool
def request_referral(user_message: str) -> List[Document]:
    """Handles requests for menu recommendations.
    
    It first searches using the user's message, and falls back to searching
    for 'popular items' if no results are found.
    """
    docs = menu_db.similarity_search(user_message, k=3)
    if not docs:
        docs = menu_db.similarity_search("인기 메뉴", k=3)

    if len(docs) > 0:
        return docs
    
    return [Document(page_content="추천할 만한 메뉴를 찾지 못했습니다.")]

@tool
def search_price() -> List[str]:
    """Searches the database for general information on prices."""
    docs = menu_db.similarity_search("메뉴 가격", k=5)

    if len(docs) > 0:
        return docs
    
    return [Document(page_content="관련 가격 정보를 찾을 수 없습니다.")]

tools = [search_menu, request_referral, search_price]
llm_with_tools = llm.bind_tools(tools)

In [21]:
def agent_node(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

tool_node = ToolNode(tools)

def should_continue(state: MessagesState) -> Literal["tools", "__end__"]:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "__end__"

builder = StateGraph(MessagesState)

builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.set_entry_point("agent")
builder.add_conditional_edges(
    "agent",
    should_continue,
)
builder.add_edge("tools", "agent")

In [23]:
memory = MemorySaver()
cafe_agent_graph = builder.compile(checkpointer=memory)

print("\n카페 메뉴 안내 에이전트가 준비되었습니다.")
print("대화를 종료하려면 'exit', 'quit', '종료'를 입력하세요.")

thread_id = str(uuid.uuid4())
thread = {"configurable": {"thread_id": thread_id}}

while True:
    print("\n" + "="*50)
    user_input = input(">> ")
    if user_input.lower() in ["exit", "quit", "종료"]:
        print("Bot: 감사합니다! 다음에 또 이용해주세요.")
        break

    print(f"사용자: {user_input}")

    final_state = cafe_agent_graph.invoke(
        {"messages": [HumanMessage(content=user_input)]}, config=thread
    )

    ai_response = final_state["messages"][-1]
    print(f"Bot: {ai_response.content}")


카페 메뉴 안내 에이전트가 준비되었습니다.
대화를 종료하려면 'exit', 'quit', '종료'를 입력하세요.

사용자: 카페 메뉴에는 뭐가 있나요?
Bot: 카페 메뉴는 다음과 같습니다:  

1. **바닐라 라떼** (₩6,000)  
   - 카페라떼에 바닐라 시럽을 더한 달콤한 메뉴. 휘핑크림 토핑으로 풍성합니다.  

2. **카페라떼** (₩5,500)  
   - 에스프레소와 스팀 우유를 조화롭게 블렌딩. 시럽/토핑 추가 가능하며 라떼 아트로 제공됩니다.  

3. **카푸치노** (₩5,000)  
   - 에스프레소, 스팀 밀크, 우유 거품의 1:1:1 비율. 계피 파우더와 함께 제공됩니다.  

4. **아메리카노** (₩4,500)  
   - 에스프레소에 뜨거운 물을 더한 클래식한 블랙 커피. 원두 본연의 풍미를 즐길 수 있습니다.  

추가로 궁금한 메뉴나 가격 정보가 있으면 알려주세요! 😊

사용자: 보내주신 메뉴 중에 제가 먹을만 한 게 있을까요? 저는 쓴 걸 잘 못 먹어서 달았으면 좋겠어요.
Bot: 쓴맛을 잘 못 드시고 단맛을 선호하신다면, **"바닐라 라떼"**를 추천드립니다!  

- **추천 이유**:  
  - 바닐라 시럽과 휘핑크림으로 달콤함이 강조되어 쓴맛이 거의 느껴지지 않습니다.  
  - 우유 비율이 높아 부드럽고 달콤한 맛을 즐기실 수 있습니다.  

> 만약 더 다양한 옵션을 원하시면, **"카페라떼"**에 **카라멜 시럽**이나 **초콜릿 시럽**을 추가해 주문해 보세요! (단, 시럽 추가 시 별도 비용이 발생할 수 있습니다.)  

혹시 다른 조건(칼로리, 유제품 제한 등)이 있다면 알려주세요! 😊

사용자: 그거 가격이 얼마인가요?
Bot: **바닐라 라떼**의 가격은 **₩6,000**입니다.  

추가로 시럽이나 토핑을 변경하실 경우 가격이 변동될 수 있으니, 주문 시 직원에게 확인해 보시는 것을 권장드립니다! 😊  

> 다른 메뉴 가격도 궁금하시다면 언제든지 물어보세요!

사용자: 오늘 날씨가 

In [24]:
try:
    current_state = cafe_agent_graph.get_state(thread)
    print("--- 현재까지의 대화 기록 ---")

    for message in current_state.values['messages']:
        print(f"- {message.type}: {message.content}")

except Exception as e:
    print(f"상태를 가져오는 데 실패했습니다: {e}")

--- 현재까지의 대화 기록 ---
- human: 카페 메뉴에는 뭐가 있나요?
- ai: [The `search_menu` function is directly necessary to answer the question about the cafe's menu items. No other functions provide menu information, and general knowledge cannot determine the specific menu.]
- tool: [
  {
    "name": "바닐라 라떼",
    "price": "₩6,000",
    "description": "카페라떼에 달콤한 바닐라 시럽을 더한 인기 메뉴입니다. 바닐라의 달콤함과 커피의 쌉싸름함이 조화롭게 어우러지며, 휘핑크림 토핑으로 더욱 풍성한 맛을 즐길 수 있습니다."
  },
  {
    "name": "카페라떼",
    "price": "₩5,500",
    "description": "진한 에스프레소에 부드럽게 스팀한 우유를 넣어 만든 대표적인 밀크 커피입니다. 크리미한 질감과 부드러운 맛이 특징이며, 다양한 시럽과 토핑 추가가 가능합니다. 라떼 아트로 시각적 즐거움도 제공합니다."
  },
  {
    "name": "카푸치노",
    "price": "₩5,000",
    "description": "에스프레소, 스팀 밀크, 우유 거품이 1:1:1 비율로 구성된 이탈리아 전통 커피입니다. 진한 커피 맛과 부드러운 우유 거품의 조화가 일품이며, 계피 파우더를 뿌려 제공합니다."
  },
  {
    "name": "아메리카노",
    "price": "₩4,500",
    "description": "진한 에스프레소에 뜨거운 물을 더해 만든 클래식한 블랙 커피입니다. 원두 본연의 맛을 가장 잘 느낄 수 있으며, 깔끔하고 깊은 풍미가 특징입니다. 설탕이나 시럽 추가 가능합니다."
  }
]
- ai: 카페 메뉴는 다음과 같습니다:  

1. **바